# 03.5 Keywords and Identifiers

Every name in your program is either a **keyword** (reserved by the language) or
an **identifier** (chosen by you). Confusing the two produces some of Python's
most baffling error messages — particularly when you accidentally hide a built-in.

## Theory

### Keywords: reserved, cannot be reused

A keyword is part of the grammar. `if`, `for`, `return`, `class` and the rest are
recognised by the tokenizer before anything else happens, so they can never be
used as names.

```python
class = 5      # SyntaxError - `class` opens a class definition
```

There are 35 keywords in Python 3.12. That is the entire reserved vocabulary of
the language, and it has stayed remarkably small.

### Soft keywords: reserved only in context

Python 3.10 introduced **soft keywords** — words that act as keywords in one
specific position but remain usable as ordinary names everywhere else.

```python
match = 5          # fine, `match` is an ordinary name here
match command:     # here it opens a match statement
    case 1: ...
```

This was deliberate: adding `match` as a hard keyword would have broken every
program that used it as a variable.

### Identifiers: the names you choose

An identifier must:

1. Start with a **letter** or **underscore** — never a digit
2. Contain only letters, digits and underscores
3. Not be a keyword

Identifiers are **case-sensitive**: `total`, `Total` and `TOTAL` are three
different names.

### The three levels of "taken"

This is the part that matters in practice:

- **Keywords** — reserved. Using one is an immediate `SyntaxError`.
- **Built-ins** — `list`, `sum`, `id`, `type`. *Not* reserved. Python will let
  you overwrite them, and then things break mysteriously.
- **Everything else** — yours.

The middle category is the dangerous one, because there is no error at the point
you make the mistake.

In [ ]:
import keyword
import sys

# keyword.kwlist holds every hard keyword in this Python version.
all_keywords = keyword.kwlist

print("Python", ".".join(str(p) for p in sys.version_info[:2]),
      "has", len(all_keywords), "hard keywords:")
print("")

# Print them in rows of six so the list is scannable.
for start in range(0, len(all_keywords), 6):
    row = all_keywords[start:start + 6]
    print("   " + "  ".join(word.ljust(9) for word in row))

# Soft keywords are reserved only in specific positions.
print("")
print("Soft keywords (reserved only in context):", keyword.softkwlist)

# There is a function to test any string.
print("")
print("Is 'for' a keyword?  ", keyword.iskeyword("for"))
print("Is 'total' a keyword?", keyword.iskeyword("total"))
print("Is 'match' a keyword?", keyword.iskeyword("match"), "- it is SOFT")

## Grouping the keywords by job

Thirty-five words is easier to learn when grouped by what they do.

In [ ]:
groups = [
    ("Control flow", ["if", "elif", "else", "for", "while", "break",
                      "continue", "pass"]),
    ("Functions", ["def", "return", "lambda", "yield"]),
    ("Classes", ["class"]),
    ("Errors", ["try", "except", "finally", "raise", "assert"]),
    ("Imports", ["import", "from", "as"]),
    ("Logic", ["and", "or", "not", "in", "is"]),
    ("Values", ["True", "False", "None"]),
    ("Scope", ["global", "nonlocal"]),
    ("Context", ["with"]),
    ("Async", ["async", "await"]),
    ("Other", ["del"]),
]

print("Group          Count  Keywords")
print("-" * 72)

total_listed = 0
for group_name, words in groups:
    total_listed += len(words)
    print(group_name.ljust(15), str(len(words)).ljust(6), " ".join(words))

print("")
print("Listed:", total_listed, "of", len(keyword.kwlist), "total")
print("(The remainder are soft keywords and a few rarely used ones.)")

## What makes a valid identifier

Python provides `str.isidentifier()` to test any string directly.

In [ ]:
import keyword

candidates = [
    "total",          # plain lowercase
    "total_price",    # snake_case
    "_private",       # leading underscore
    "__dunder__",     # double underscore both ends
    "Total",          # capitalised
    "TOTAL",          # all caps
    "item2",          # digits are fine, just not first
    "2item",          # starts with a digit
    "total-price",    # hyphen reads as minus
    "total price",    # space
    "class",          # a keyword
    "match",          # a SOFT keyword
    "list",           # a built-in, not a keyword
    "café",           # non-ASCII letters are allowed
    "",               # empty
]

print("Candidate        Valid?  Keyword?  Verdict")
print("-" * 62)

for candidate in candidates:
    # isidentifier() checks the character rules only.
    valid_chars = candidate.isidentifier()

    # A valid identifier can still be rejected for being a keyword.
    is_kw = keyword.iskeyword(candidate)

    if not valid_chars:
        verdict = "invalid characters"
    elif is_kw:
        verdict = "reserved - SyntaxError"
    else:
        verdict = "usable"

    print(repr(candidate).ljust(16), str(valid_chars).ljust(7),
          str(is_kw).ljust(9), verdict)

## The dangerous middle ground: shadowing built-ins

Built-in names are **not reserved**. Python lets you reassign them without
complaint, and the failure surfaces later, somewhere else, with a confusing
message.

In [ ]:
# There are around 150 built-in names available with no import at all.
import builtins

builtin_names = [name for name in dir(builtins) if not name.startswith("_")]
print("Built-in names available:", len(builtin_names))

# The ones people shadow by accident, because they are natural variable names.
commonly_shadowed = ["list", "dict", "set", "str", "int", "type", "id",
                     "sum", "max", "min", "input", "next", "filter", "map",
                     "object", "bytes", "hash", "format", "all", "any"]

print("")
print("Commonly shadowed by accident:")
for start in range(0, len(commonly_shadowed), 5):
    row = commonly_shadowed[start:start + 5]
    print("   " + "  ".join(name.ljust(9) for name in row))

print("")
print("None of these are keywords, so Python allows the assignment:")
print("   keyword.iskeyword('list') ->", keyword.iskeyword("list"))

In [ ]:
def demonstrate_shadowing():
    """Show a shadowed built-in failing, contained inside one function."""
    # This looks completely reasonable.
    list = ["apple", "banana"]
    print("   the variable works fine:", list)

    # But the built-in is now hidden inside this scope.
    try:
        list("abc")
    except TypeError as error:
        print("   calling list('abc') now fails:", error)


def demonstrate_shadowing_sum():
    """The same problem with sum, which is even easier to hit."""
    # A natural name for a running total.
    sum = 0
    for value in [1, 2, 3]:
        sum = sum + value
    print("   the total computed fine:", sum)

    try:
        sum([4, 5, 6])
    except TypeError as error:
        print("   but sum([4, 5, 6]) now fails:", error)


print("Shadowing `list`:")
demonstrate_shadowing()

print("")
print("Shadowing `sum`:")
demonstrate_shadowing_sum()

print("")
print("Outside those functions the built-ins are untouched:")
print("   list('abc') ->", list("abc"))
print("   sum([1,2,3]) ->", sum([1, 2, 3]))

### How to avoid it

Add a trailing underscore, or pick a more descriptive name. A more descriptive
name is almost always better anyway.

In [ ]:
# The fixes, in order of preference.
fixes = [
    ("list = [...]", "items = [...]", "describes the contents"),
    ("sum = 0", "running_total = 0", "describes the role"),
    ("input = get()", "user_input = get()", "describes the source"),
    ("type = 'admin'", "account_type = 'admin'", "describes the subject"),
    ("id = 42", "user_id = 42", "describes what it identifies"),
    ("str = 'x'", "text = 'x'", "a plain, accurate word"),
    ("class = 'A'", "class_ = 'A'", "trailing underscore - keyword, no choice"),
]

print("Instead of          Write               Because")
print("-" * 68)
for bad, good, reason in fixes:
    print(bad.ljust(20), good.ljust(20), reason)

print("")
print("The trailing underscore is PEP 8's official escape hatch, but it")
print("is a last resort - reach for a better name first.")

## Underscore conventions

Underscores carry meaning in Python. These are conventions, not rules — but they
are universally understood, and two of them do change behaviour.

In [ ]:
conventions = [
    ("name", "public", "normal, part of your API", "no"),
    ("_name", "internal", "a hint: do not rely on this", "no"),
    ("__name", "name-mangled", "rewritten inside classes", "YES"),
    ("__name__", "dunder", "reserved by Python - do not invent your own", "no"),
    ("name_", "avoid clash", "when the good name is a keyword", "no"),
    ("_", "throwaway", "a value you do not intend to use", "no"),
]

print("Form         Called         Means                                  Changes behaviour?")
print("-" * 92)
for form, called, means, changes in conventions:
    print(form.ljust(12), called.ljust(14), means.ljust(38), changes)


# The throwaway underscore, in practice.
print("")
print("The throwaway _ in use:")

coordinates = [(1, 2), (3, 4), (5, 6)]

# We only care about the second value of each pair.
for _, y_value in coordinates:
    print("   y is", y_value)

# Also common for a loop counter you never reference.
print("")
for _ in range(3):
    print("   repeating without needing the number")

In [ ]:
# Name mangling is the one convention that genuinely changes behaviour.

class Account:
    """Demonstrate what a double leading underscore does."""

    def __init__(self):
        self.public = "anyone can read this"
        self._internal = "please do not rely on this"
        self.__mangled = "Python rewrites this name"

    def read_mangled(self):
        """Read the mangled attribute from inside the class."""
        # Inside the class, the original name works.
        return self.__mangled


account = Account()

print("Attributes Python actually stored:")
for attribute in vars(account):
    print("   ", attribute)

print("")
print("Notice __mangled became _Account__mangled.")
print("")
print("Reading it from inside the class works:", account.read_mangled())

# From outside, the original name no longer exists.
try:
    account.__mangled
except AttributeError as error:
    print("From outside:", error)

# The mangled name is still reachable - this is a convention, not security.
print("Via the mangled name:", account._Account__mangled)

print("")
print("Purpose: avoid accidental clashes in subclasses. Chapter 25 covers it.")

## Naming conventions from PEP 8

Consistent naming is what makes unfamiliar code readable. These conventions are
followed by effectively every Python project.

In [ ]:
naming_rules = [
    ("Variables", "snake_case", "user_name, total_price"),
    ("Functions", "snake_case", "calculate_total(), send_email()"),
    ("Constants", "UPPER_SNAKE_CASE", "MAX_RETRIES, DEFAULT_TIMEOUT"),
    ("Classes", "PascalCase", "BankAccount, HttpClient"),
    ("Modules", "lowercase", "utils.py, data_loader.py"),
    ("Packages", "lowercase", "myproject, requests"),
    ("Exceptions", "PascalCase + Error", "ValidationError, TimeoutError"),
    ("Type variables", "short PascalCase", "T, KeyType"),
    ("Private", "_leading_underscore", "_cache, _connect()"),
]

print("What            Convention             Example")
print("-" * 70)
for what, convention, example in naming_rules:
    print(what.ljust(16), convention.ljust(22), example)

print("")
print("Two rules worth stating plainly:")
print("   - Never use camelCase for variables. That is Java, not Python.")
print("   - Never use single letters except for a genuine index or coordinate.")

## Choosing good names

A name is documentation that cannot go stale. These are the patterns that make
code read well.

In [ ]:
comparisons = [
    ("d", "days_until_expiry", "a letter tells the reader nothing"),
    ("data", "customer_records", "'data' is true of everything"),
    ("temp", "celsius_reading", "'temp' means temperature or temporary"),
    ("flag", "is_email_verified", "say what the flag means"),
    ("process()", "validate_payment()", "'process' describes no specific action"),
    ("lst", "pending_orders", "abbreviations save nothing"),
    ("x1, x2", "start_date, end_date", "numbered names hide the difference"),
    ("check()", "raises_on_invalid()", "say what happens, not that something happens"),
]

print("Poor              Better                  Why")
print("-" * 76)
for poor, better, why in comparisons:
    print(poor.ljust(18), better.ljust(24), why)

print("")
print("Conventions for boolean names:")
booleans = [
    ("is_", "is_active, is_valid", "a state"),
    ("has_", "has_permission, has_items", "possession"),
    ("can_", "can_edit, can_retry", "capability"),
    ("should_", "should_retry, should_log", "a decision"),
]
for prefix, example, meaning in booleans:
    print("   ", prefix.ljust(8), example.ljust(28), meaning)

## Unicode identifiers

Python 3 allows non-ASCII letters in names. This is genuinely useful for
non-English codebases, but comes with a subtle trap: Python normalises names
using NFKC, so two visually different names can be the *same* identifier.

In [ ]:
# Non-ASCII identifiers are legal.
café = "espresso"
数量 = 42
переменная = "value"

print("Non-ASCII names work:")
print("   café =", café)
print("   数量 =", 数量)
print("   переменная =", переменная)

# But Python normalises identifiers with NFKC before comparing them.
# These two source spellings become the SAME name.
import unicodedata

name_one = "x"                              # ordinary latin x
name_two = unicodedata.normalize("NFKC", "ｘ")   # fullwidth x

print("")
print("Fullwidth x normalises to:", repr(name_two))
print("Same identifier as plain x?", name_one == name_two)

print("")
print("Practical advice: stick to ASCII for anything shared or published.")
print("Mixed scripts in identifiers are a known source of confusion.")

## Takeaways

1. **35 keywords** are reserved — using one as a name is an immediate
   `SyntaxError`.
2. **Soft keywords** (`match`, `case`, `type`, `_`) are reserved only in
   context, so existing code did not break when they were added.
3. Identifiers start with a letter or underscore, never a digit, and are
   **case-sensitive**.
4. **Built-ins are not reserved.** Shadowing `list`, `sum` or `type` is allowed
   and fails later with a confusing message.
5. Fix shadowing with a **more descriptive name**; a trailing underscore is the
   last resort.
6. `__double_leading` triggers **name mangling** inside classes — the only
   underscore convention that changes behaviour.
7. `snake_case` for variables and functions, `PascalCase` for classes,
   `UPPER_SNAKE_CASE` for constants.
8. Boolean names read best with `is_`, `has_`, `can_` or `should_`.

## Try it yourself

1. Run `keyword.kwlist` and count how many of the 35 you already recognise.
2. In a fresh cell, assign `type = "admin"`, then call `type(5)`. Read the error,
   then restart the kernel to recover.
3. Test five names of your own with `str.isidentifier()`. Any surprises?
4. Create a class with a `__private` attribute and inspect it with `vars()`.
   What name did Python actually store?